# U-Net patch + overlap-tile 继续加训实验

从上一轮 patch/overlap-tile 的最佳权重继续训练，用于当前 IoU 尚未收敛的情况。


## 1. 实验配置

如果服务器已经解压好数据，保持 `DOWNLOAD_DATA = False`；如果没有数据，改成 `True` 后运行下载。

In [ ]:
from pathlib import Path

DOWNLOAD_DATA = False
RUN_TRAIN = True
RUN_PREDICT = True
RUN_TRAIN_VISUALIZATION = True

DATA_URL = 'https://ascend-professional-construction-dataset.obs.cn-north-4.myhuaweicloud.com:443/20230927/Unet.zip'
ZIP_PATH = Path('Unet.zip')
PROJECT_ROOT = Path('Unet')
DATA_ROOT = PROJECT_ROOT / 'ISBI'
TRAIN_DIR = DATA_ROOT / 'train'
VAL_DIR = DATA_ROOT / 'val'
TEST_DIR = DATA_ROOT / 'test'

OUTPUT_DIR = Path('output_patch_overlap_tile_continue_more')
PRED_DIR = OUTPUT_DIR / 'predict'
FIG_DIR = OUTPUT_DIR / 'figures'
CKPT_DIR = OUTPUT_DIR / 'checkpoint'
TRAIN_VIZ_DIR = OUTPUT_DIR / 'train_visualization'
for path in [OUTPUT_DIR, PRED_DIR, FIG_DIR, CKPT_DIR, TRAIN_VIZ_DIR]:
    path.mkdir(parents=True, exist_ok=True)

# 原论文式 overlap-tile 配置：输入 patch 互相重叠，只使用中心区域拼回原图。
PATCH_SIZE = 256
CENTER_SIZE = 128
CONTEXT_SIZE = (PATCH_SIZE - CENTER_SIZE) // 2
TILE_BATCH_SIZE = 8
PATCHES_PER_IMAGE_PER_EPOCH = 8

# 为了兼容旧函数名，IMG_SIZE 等于 patch 输入尺寸；本 notebook 不再做整图 resize。
IMG_SIZE = PATCH_SIZE
BATCH_SIZE = 4
EPOCHS = 160
LR = 2e-5
EARLY_STOP_PATIENCE = None
EARLY_STOP_MIN_DELTA = 1e-4
SEED = 1
INVERT_MASK = True
USE_PRETRAINED = True
TRAIN_FINAL_LAYER_ONLY = False
PRETRAINED_CKPT = Path('output_patch_overlap_tile_continue/checkpoint/best_composite.ckpt')
FALLBACK_CKPT = Path('output_patch_overlap_tile_continue/checkpoint/best_iou.ckpt')
WEIGHT_MAP_CKPT = Path('output_weight_map_loss_continue/checkpoint/best_UNet_weight_map_loss.ckpt')
HANDOUT_BEST_CKPT = PROJECT_ROOT / 'checkpoint' / 'best_UNet.ckpt'
USE_HANDOUT_BEST_FOR_PREDICT = False

INITIAL_CKPT = CKPT_DIR / 'initial_pretrained.ckpt'
LAST_CKPT = CKPT_DIR / 'last_epoch.ckpt'
BEST_IOU_CKPT = CKPT_DIR / 'best_iou.ckpt'
BEST_DICE_CKPT = CKPT_DIR / 'best_dice.ckpt'
BEST_PRECISION_CKPT = CKPT_DIR / 'best_precision.ckpt'
BEST_SPEC_CKPT = CKPT_DIR / 'best_specificity.ckpt'
BEST_LOSS_CKPT = CKPT_DIR / 'best_val_loss.ckpt'
BEST_COMPOSITE_CKPT = CKPT_DIR / 'best_composite.ckpt'
BEST_CKPT = BEST_COMPOSITE_CKPT

USE_ELASTIC_AUG = True
ELASTIC_PROB = 0.5
ELASTIC_ALPHA = 24
ELASTIC_SIGMA = 4
ROT90_PROB = 0.75
USE_RANDOM_FLIP = True
SHOW_GETITEM_AUGMENTATION = False
GETITEM_AUG_SAMPLE_INDEX = 0
GETITEM_AUG_REPEATS = 10

WEIGHTED_BCE_WEIGHT = 0.4
FOCAL_TVERSKY_WEIGHT = 0.6
TVERSKY_ALPHA = 0.70
TVERSKY_BETA = 0.30
FOCAL_TVERSKY_GAMMA = 1.33

WEIGHT_MAP_W0 = 7.0
WEIGHT_MAP_SIGMA = 3.0
WEIGHT_MAP_NEG_CLASS_WEIGHT = 1.0
WEIGHT_MAP_POS_CLASS_WEIGHT = 4.0
WEIGHT_MAP_CLIP_MAX = 20.0

COMPOSITE_WEIGHTS = {
    'iou': 0.40,
    'dice': 0.20,
    'precision': 0.25,
    'spec': 0.15,
}
TRAIN_VIZ_TOPK = 5

if PATCH_SIZE <= CENTER_SIZE or (PATCH_SIZE - CENTER_SIZE) % 2 != 0:
    raise ValueError('PATCH_SIZE must be larger than CENTER_SIZE and have even margin.')
if PATCH_SIZE % 16 != 0:
    raise ValueError('PATCH_SIZE should be divisible by 16 for the current 4-level U-Net.')

print('train:', TRAIN_DIR)
print('val:', VAL_DIR)
print('test:', TEST_DIR)
print('output:', OUTPUT_DIR)
print('continue_from:', PRETRAINED_CKPT)
print('fallback patch best_iou ckpt:', FALLBACK_CKPT)
print('fallback weight_map ckpt:', WEIGHT_MAP_CKPT)
print('target:', 'boundary/membrane positive' if INVERT_MASK else 'white region positive')
print('epochs:', EPOCHS, 'lr:', LR, 'early_stop_patience:', EARLY_STOP_PATIENCE)
print('overlap_tile:', {
    'patch_size': PATCH_SIZE,
    'center_size': CENTER_SIZE,
    'context_size': CONTEXT_SIZE,
    'tile_batch_size': TILE_BATCH_SIZE,
    'patches_per_image_per_epoch': PATCHES_PER_IMAGE_PER_EPOCH,
    'tiles_for_512x512': (512 // CENTER_SIZE) * (512 // CENTER_SIZE),
})
print('augment:', {
    'flip': USE_RANDOM_FLIP,
    'elastic': USE_ELASTIC_AUG,
    'elastic_prob': ELASTIC_PROB,
    'elastic_alpha': ELASTIC_ALPHA,
    'elastic_sigma': ELASTIC_SIGMA,
    'rot90_prob': ROT90_PROB,
})
print('loss_config:', {
    'weighted_bce_weight': WEIGHTED_BCE_WEIGHT,
    'focal_tversky_weight': FOCAL_TVERSKY_WEIGHT,
    'tversky_alpha_fp': TVERSKY_ALPHA,
    'tversky_beta_fn': TVERSKY_BETA,
    'focal_tversky_gamma': FOCAL_TVERSKY_GAMMA,
    'weight_map_w0': WEIGHT_MAP_W0,
    'weight_map_sigma': WEIGHT_MAP_SIGMA,
    'weight_map_pos_class_weight': WEIGHT_MAP_POS_CLASS_WEIGHT,
    'composite_weights': COMPOSITE_WEIGHTS,
})


In [ ]:
if DOWNLOAD_DATA and not PROJECT_ROOT.exists():
    import urllib.request
    import zipfile

    print('Downloading:', DATA_URL)
    urllib.request.urlretrieve(DATA_URL, ZIP_PATH)
    print('Unzipping:', ZIP_PATH)
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall('.')

for required in [TRAIN_DIR, VAL_DIR]:
    if not required.exists():
        raise FileNotFoundError(f'Missing data directory: {required}. 请先下载并解压 Unet.zip。')

## 2. 导入环境

In [ ]:
import csv
import math
import os
import random
from typing import Iterable

import numpy as np
from PIL import Image, ImageSequence
import matplotlib.pyplot as plt

import mindspore
import mindspore.dataset as ds
import mindspore.dataset.vision as vision
import mindspore.nn as nn
import mindspore.ops as ops
from mindspore import Tensor, load_checkpoint, save_checkpoint
from mindspore.dataset.vision import Inter

try:
    from scipy.ndimage import gaussian_filter, map_coordinates, distance_transform_edt
except ImportError:
    gaussian_filter = None
    map_coordinates = None
    distance_transform_edt = None

mindspore.set_context(device_target='Ascend', save_graphs=False)
mindspore.set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

print('MindSpore:', mindspore.__version__)
if USE_ELASTIC_AUG and (gaussian_filter is None or map_coordinates is None):
    raise ImportError('elastic deformation 需要 scipy，请在环境中安装 scipy，或设置 USE_ELASTIC_AUG=False')
if distance_transform_edt is None:
    raise ImportError('weight map loss 需要 scipy.ndimage.distance_transform_edt，请安装 scipy。')


## 3. 数据查看

In [ ]:
def show_images(image_list, num=6, title=''):
    images = list(image_list)[:num]
    if not images:
        print('No images to show:', title)
        return
    row = 3 if len(images) > 6 else 2 if len(images) > 3 else 1
    col = math.ceil(len(images) / row)
    plt.figure(figsize=(3 * col, 3 * row))
    for i, img in enumerate(images):
        plt.subplot(row, col, i + 1)
        plt.imshow(img, cmap='gray')
        plt.title(str(i))
        plt.xticks([])
        plt.yticks([])
    if title:
        plt.suptitle(title)
    plt.tight_layout()
    plt.show()

train_volume_path = PROJECT_ROOT / 'data' / 'train-volume.tif'
train_masks_path = PROJECT_ROOT / 'data' / 'train-labels.tif'
if train_volume_path.exists() and train_masks_path.exists():
    train_volume = np.array([np.array(p) for p in ImageSequence.Iterator(Image.open(train_volume_path))])
    train_masks = np.array([np.array(p) for p in ImageSequence.Iterator(Image.open(train_masks_path))])
    show_images(train_volume, num=12, title='train images')
    show_images(train_masks, num=12, title='train masks')
else:
    print('TIFF preview files not found, skip preview.')

## 4. 原分辨率 patch 数据读取与 overlap-tile 工具


In [ ]:
def normalize_chw(image):
    arr = image.astype(np.float32) / 255.0
    return np.transpose(arr, (2, 0, 1))


def mask_to_chw(mask):
    arr = (mask.astype(np.float32) / 255.0)
    return arr[None, :, :]


def weight_to_chw(weight_map):
    return weight_map.astype(np.float32)[None, :, :]


def make_weight_map(label):
    boundary = label > 127
    class_weight = np.where(boundary, WEIGHT_MAP_POS_CLASS_WEIGHT, WEIGHT_MAP_NEG_CLASS_WEIGHT).astype(np.float32)
    distance_to_boundary = distance_transform_edt(~boundary).astype(np.float32)
    boundary_bonus = WEIGHT_MAP_W0 * np.exp(-(distance_to_boundary ** 2) / (2.0 * WEIGHT_MAP_SIGMA ** 2))
    weight_map = class_weight + boundary_bonus.astype(np.float32)
    return np.clip(weight_map, 0.0, WEIGHT_MAP_CLIP_MAX).astype(np.float32)


def load_image_mask(image_path, mask_path=None):
    image = np.array(Image.open(image_path).convert('RGB'))
    if mask_path is None:
        return image, None
    label = np.array(Image.open(mask_path).convert('L'))
    if INVERT_MASK:
        label = 255 - label
    label = ((label > 127).astype(np.uint8) * 255)
    return image, label


def pad_reflect_to_min_size(image, label=None, min_size=PATCH_SIZE):
    h, w = image.shape[:2]
    pad_h = max(0, min_size - h)
    pad_w = max(0, min_size - w)
    if pad_h == 0 and pad_w == 0:
        return image, label
    pad_top = pad_h // 2
    pad_bottom = pad_h - pad_top
    pad_left = pad_w // 2
    pad_right = pad_w - pad_left
    image = np.pad(image, ((pad_top, pad_bottom), (pad_left, pad_right), (0, 0)), mode='reflect')
    if label is not None:
        label = np.pad(label, ((pad_top, pad_bottom), (pad_left, pad_right)), mode='reflect')
    return image, label


def random_patch_crop(image, label, patch_size=PATCH_SIZE):
    image, label = pad_reflect_to_min_size(image, label, patch_size)
    h, w = image.shape[:2]
    y0 = random.randint(0, h - patch_size)
    x0 = random.randint(0, w - patch_size)
    image_patch = image[y0:y0 + patch_size, x0:x0 + patch_size]
    label_patch = label[y0:y0 + patch_size, x0:x0 + patch_size]
    return image_patch, label_patch


def augment_patch(image, label):
    if USE_RANDOM_FLIP and random.random() < 0.5:
        image = np.ascontiguousarray(np.flip(image, axis=1))
        label = np.ascontiguousarray(np.flip(label, axis=1))
    if USE_RANDOM_FLIP and random.random() < 0.5:
        image = np.ascontiguousarray(np.flip(image, axis=0))
        label = np.ascontiguousarray(np.flip(label, axis=0))
    if random.random() < ROT90_PROB:
        k = random.randint(1, 3)
        image = np.ascontiguousarray(np.rot90(image, k))
        label = np.ascontiguousarray(np.rot90(label, k))
    if USE_ELASTIC_AUG and random.random() < ELASTIC_PROB:
        image, label_3d = elastic_transform(image, label[:, :, None], alpha=ELASTIC_ALPHA, sigma=ELASTIC_SIGMA)
        label = label_3d[:, :, 0]
    return image, label


def elastic_transform(image, label, alpha=24, sigma=4):
    if gaussian_filter is None or map_coordinates is None:
        raise ImportError('elastic deformation 需要 scipy，请安装 scipy，或设置 USE_ELASTIC_AUG=False')

    h, w = image.shape[:2]
    dx = gaussian_filter((np.random.rand(h, w) * 2 - 1), sigma, mode='reflect') * alpha
    dy = gaussian_filter((np.random.rand(h, w) * 2 - 1), sigma, mode='reflect') * alpha
    x, y = np.meshgrid(np.arange(w), np.arange(h))
    indices = ((y + dy).reshape(-1), (x + dx).reshape(-1))

    warped_image = np.empty_like(image)
    for c in range(image.shape[2]):
        channel = map_coordinates(image[..., c], indices, order=1, mode='reflect').reshape(h, w)
        warped_image[..., c] = np.clip(channel, 0, 255).astype(image.dtype)

    warped_label = map_coordinates(label[..., 0], indices, order=0, mode='reflect').reshape(h, w)
    warped_label = (warped_label > 127).astype(label.dtype) * 255
    return warped_image, warped_label[:, :, None]


class SegmentationPatchDataset:
    def __init__(self, data_dir, augment=False, patches_per_image=1, have_mask=True):
        self.data_dir = Path(data_dir)
        self.augment = augment
        self.patches_per_image = patches_per_image
        self.have_mask = have_mask
        image_paths = sorted((self.data_dir / 'image').glob('*.png'))
        if self.have_mask:
            mask_paths = sorted((self.data_dir / 'mask').glob('*.png'))
            mask_by_name = {path.stem: path for path in mask_paths}
            pairs = [(img, mask_by_name[img.stem]) for img in image_paths if img.stem in mask_by_name]
            missing_masks = [img.name for img in image_paths if img.stem not in mask_by_name]
            extra_masks = [mask.name for mask in mask_paths if mask.stem not in {img.stem for img in image_paths}]
            if missing_masks:
                raise ValueError(f'这些 image 没有对应 mask: {missing_masks[:10]}')
            if extra_masks:
                print(f'忽略没有对应 image 的 mask: {extra_masks[:10]}')
            self.image_paths = [img for img, _ in pairs]
            self.mask_paths = [mask for _, mask in pairs]
        else:
            self.image_paths = image_paths
            self.mask_paths = [None] * len(self.image_paths)

    def __len__(self):
        return len(self.image_paths) * self.patches_per_image

    @property
    def column_names(self):
        return ['image', 'label', 'weight_map']

    def __getitem__(self, index):
        real_index = index % len(self.image_paths)
        image, label = load_image_mask(self.image_paths[real_index], self.mask_paths[real_index])
        image, label = random_patch_crop(image, label, PATCH_SIZE)
        if self.augment:
            image, label = augment_patch(image, label)
        weight_map = make_weight_map(label)
        return normalize_chw(image), mask_to_chw(label), weight_to_chw(weight_map)


def create_patch_dataset(data_dir, batch_size, augment=False, shuffle=True, patches_per_image=1, have_mask=True):
    source = SegmentationPatchDataset(data_dir, augment=augment, patches_per_image=patches_per_image, have_mask=have_mask)
    dataset = ds.GeneratorDataset(source, column_names=source.column_names, shuffle=shuffle)
    dataset = dataset.batch(batch_size)
    return dataset


def iter_image_mask_paths(data_dir):
    data_dir = Path(data_dir)
    image_paths = sorted((data_dir / 'image').glob('*.png'))
    mask_dir = data_dir / 'mask'
    for image_path in image_paths:
        mask_path = mask_dir / image_path.name
        if mask_path.exists():
            yield image_path, mask_path


def tile_positions(length, center_size=CENTER_SIZE):
    positions = list(range(0, length, center_size))
    if not positions or positions[-1] != length - min(center_size, length - positions[-1]):
        pass
    return positions


def predict_overlap_tile_array(model, image, patch_size=PATCH_SIZE, center_size=CENTER_SIZE, batch_size=TILE_BATCH_SIZE):
    context = (patch_size - center_size) // 2
    h, w = image.shape[:2]
    padded = np.pad(image, ((context, context), (context, context), (0, 0)), mode='reflect')
    prob = np.zeros((h, w), dtype=np.float32)

    tasks = []
    for y in range(0, h, center_size):
        for x in range(0, w, center_size):
            center_h = min(center_size, h - y)
            center_w = min(center_size, w - x)
            patch = padded[y:y + patch_size, x:x + patch_size]
            tasks.append((y, x, center_h, center_w, normalize_chw(patch)))

    model.set_train(False)
    for start in range(0, len(tasks), batch_size):
        batch_tasks = tasks[start:start + batch_size]
        batch = np.stack([task[4] for task in batch_tasks]).astype(np.float32)
        pred_batch = model(Tensor(batch, mindspore.float32)).asnumpy()[:, 0]
        for pred, (y, x, center_h, center_w, _) in zip(pred_batch, batch_tasks):
            center_pred = pred[context:context + center_h, context:context + center_w]
            prob[y:y + center_h, x:x + center_w] = center_pred
    return prob


def fullres_label_and_weight(mask_path):
    _, label = load_image_mask(mask_path.parent.parent / 'image' / mask_path.name, mask_path)
    weight_map = make_weight_map(label)
    label_chw = mask_to_chw(label).astype(np.float32)
    weight_chw = weight_to_chw(weight_map).astype(np.float32)
    return label_chw, weight_chw


train_dataset = create_patch_dataset(
    TRAIN_DIR,
    batch_size=BATCH_SIZE,
    augment=True,
    shuffle=True,
    patches_per_image=PATCHES_PER_IMAGE_PER_EPOCH,
    have_mask=True,
)

print('train image count:', len(SegmentationPatchDataset(TRAIN_DIR, have_mask=True).image_paths))
print('train patch samples per epoch:', train_dataset.get_dataset_size() * BATCH_SIZE)
print('patch shape:', PATCH_SIZE, 'center kept during inference:', CENTER_SIZE, 'context:', CONTEXT_SIZE)


### 4.1 单样本在线数据增强可视化


In [ ]:
def visualize_getitem_augmentations(data_dir=TRAIN_DIR, sample_index=0, repeats=10, out_file=None):
    base_source = SegmentationPatchDataset(data_dir, augment=False, patches_per_image=1, have_mask=True)
    aug_source = SegmentationPatchDataset(data_dir, augment=True, patches_per_image=max(repeats, 1), have_mask=True)
    if len(base_source) == 0:
        print('No samples found:', data_dir)
        return

    sample_index = sample_index % len(base_source)
    base_image, base_label, base_weight = base_source[sample_index]
    base_image = np.transpose(base_image, (1, 2, 0))
    base_label = base_label[0]

    cols = repeats + 1
    fig, axes = plt.subplots(2, cols, figsize=(2.2 * cols, 4.8))
    axes[0, 0].imshow(base_image)
    axes[0, 0].set_title(f'patch\n{base_image.shape[0]}x{base_image.shape[1]}', fontsize=8)
    axes[1, 0].imshow(base_label, cmap='gray', vmin=0, vmax=1)
    axes[1, 0].set_title('mask', fontsize=8)

    for i in range(repeats):
        image, label, weight_map = aug_source[sample_index + i]
        image = np.transpose(image, (1, 2, 0))
        label = label[0]
        axes[0, i + 1].imshow(image)
        axes[0, i + 1].set_title(f'aug {i + 1}\n{image.shape[0]}x{image.shape[1]}', fontsize=8)
        axes[1, i + 1].imshow(label, cmap='gray', vmin=0, vmax=1)
        axes[1, i + 1].set_title('mask', fontsize=8)

    for ax in axes.ravel():
        ax.axis('off')
    plt.tight_layout()
    if out_file is not None:
        plt.savefig(out_file, dpi=160, bbox_inches='tight')
        print('saved augmentation figure:', out_file)
    plt.show()


if SHOW_GETITEM_AUGMENTATION:
    visualize_getitem_augmentations(
        TRAIN_DIR,
        sample_index=GETITEM_AUG_SAMPLE_INDEX,
        repeats=GETITEM_AUG_REPEATS,
        out_file=FIG_DIR / 'getitem_patch_augmentation_examples.png',
    )


In [ ]:
for data, label, weight_map in train_dataset.create_tuple_iterator():
    print('train patch image:', data.shape, data.dtype)
    print('train patch label:', label.shape, label.dtype)
    print('train patch weight_map:', weight_map.shape, weight_map.dtype, 'min=', float(weight_map.asnumpy().min()), 'max=', float(weight_map.asnumpy().max()))
    break

example_val_image = next((VAL_DIR / 'image').glob('*.png'))
if example_val_image:
    example_arr = np.array(Image.open(example_val_image).convert('RGB'))
    example_prob = np.zeros(example_arr.shape[:2], dtype=np.float32)
    tile_count = len(range(0, example_arr.shape[0], CENTER_SIZE)) * len(range(0, example_arr.shape[1], CENTER_SIZE))
    print('example full image:', example_arr.shape, 'overlap-tile patch count:', tile_count)


## 5. U-Net 网络结构

In [ ]:
def double_conv(in_ch, out_ch):
    return nn.SequentialCell(
        nn.Conv2d(in_ch, out_ch, kernel_size=3, pad_mode='same'),
        nn.BatchNorm2d(out_ch),
        nn.ReLU(),
        nn.Conv2d(out_ch, out_ch, kernel_size=3, pad_mode='same'),
        nn.BatchNorm2d(out_ch),
        nn.ReLU(),
    )


class UNet(nn.Cell):
    def __init__(self, in_ch=3, n_classes=1):
        super().__init__()
        self.double_conv1 = double_conv(in_ch, 64)
        self.maxpool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.double_conv2 = double_conv(64, 128)
        self.maxpool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.double_conv3 = double_conv(128, 256)
        self.maxpool3 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.double_conv4 = double_conv(256, 512)
        self.maxpool4 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.double_conv5 = double_conv(512, 1024)
        self.upsample1 = nn.ResizeBilinear()
        self.double_conv6 = double_conv(1024 + 512, 512)
        self.upsample2 = nn.ResizeBilinear()
        self.double_conv7 = double_conv(512 + 256, 256)
        self.upsample3 = nn.ResizeBilinear()
        self.double_conv8 = double_conv(256 + 128, 128)
        self.upsample4 = nn.ResizeBilinear()
        self.double_conv9 = double_conv(128 + 64, 64)
        self.final = nn.Conv2d(64, n_classes, kernel_size=1, pad_mode='same')
        self.sigmoid = ops.Sigmoid()

    def construct(self, x):
        feature1 = self.double_conv1(x)
        feature2 = self.double_conv2(self.maxpool1(feature1))
        feature3 = self.double_conv3(self.maxpool2(feature2))
        feature4 = self.double_conv4(self.maxpool3(feature3))
        feature5 = self.double_conv5(self.maxpool4(feature4))

        x = self.upsample1(feature5, scale_factor=2)
        x = self.double_conv6(ops.concat((feature4, x), axis=1))
        x = self.upsample2(x, scale_factor=2)
        x = self.double_conv7(ops.concat((feature3, x), axis=1))
        x = self.upsample3(x, scale_factor=2)
        x = self.double_conv8(ops.concat((feature2, x), axis=1))
        x = self.upsample4(x, scale_factor=2)
        x = self.double_conv9(ops.concat((feature1, x), axis=1))
        return self.sigmoid(self.final(x))


net = UNet(in_ch=3, n_classes=1)
print(net)

## 6. 评价指标

In [ ]:
class SegmentationMetrics:
    def __init__(self, smooth=1e-5):
        self.smooth = smooth
        self.clear()

    def clear(self):
        self.acc = 0.0
        self.iou = 0.0
        self.dice = 0.0
        self.sens = 0.0
        self.spec = 0.0
        self.precision = 0.0
        self.fp_ratio = 0.0
        self.fn_ratio = 0.0
        self.samples = 0

    @staticmethod
    def _to_numpy(x):
        if isinstance(x, Tensor):
            return x.asnumpy()
        return np.asarray(x)

    def update(self, y_pred, y_true):
        y_pred = self._to_numpy(y_pred)
        y_true = self._to_numpy(y_true)
        y_pred = (y_pred > 0.5).astype(np.float32)
        y_true = (y_true > 0.5).astype(np.float32)
        if y_pred.shape != y_true.shape:
            raise ValueError(f'shape mismatch: pred={y_pred.shape}, true={y_true.shape}')

        for pred, true in zip(y_pred, y_true):
            pred = pred.reshape(-1)
            true = true.reshape(-1)
            tp = np.sum(pred * true)
            tn = np.sum((1 - pred) * (1 - true))
            fp = np.sum(pred * (1 - true))
            fn = np.sum((1 - pred) * true)
            union = tp + fp + fn

            self.acc += float((tp + tn) / (tp + tn + fp + fn + self.smooth))
            self.iou += float(tp / (union + self.smooth))
            self.dice += float((2 * tp) / (2 * tp + fp + fn + self.smooth))
            self.sens += float(tp / (tp + fn + self.smooth))
            self.spec += float(tn / (tn + fp + self.smooth))
            self.precision += float(tp / (tp + fp + self.smooth))
            self.fp_ratio += float(fp / (union + self.smooth))
            self.fn_ratio += float(fn / (union + self.smooth))
            self.samples += 1

    def eval(self):
        if self.samples == 0:
            raise RuntimeError('No samples were added to metrics.')
        return {
            'acc': self.acc / self.samples,
            'iou': self.iou / self.samples,
            'dice': self.dice / self.samples,
            'sens': self.sens / self.samples,
            'spec': self.spec / self.samples,
            'precision': self.precision / self.samples,
            'fp_ratio': self.fp_ratio / self.samples,
            'fn_ratio': self.fn_ratio / self.samples,
        }


def composite_score(metrics):
    return sum(COMPOSITE_WEIGHTS[k] * metrics[k] for k in COMPOSITE_WEIGHTS)


metric_test = SegmentationMetrics()
metric_test.update(
    np.array([[[[0.2, 0.5, 0.7], [0.3, 0.1, 0.2], [0.9, 0.6, 0.8]]]]),
    np.array([[[[0, 1, 1], [1, 0, 0], [0, 1, 1]]]]),
)
print(metric_test.eval())


## 7. 训练与验证

In [ ]:
class PixelWeightMapBCEFocalTverskyLoss(nn.Cell):
    def __init__(self, bce_weight=0.4, focal_tversky_weight=0.6, alpha=0.7, beta=0.3, gamma=1.33, smooth=1e-5):
        super().__init__()
        self.bce_weight = bce_weight
        self.focal_tversky_weight = focal_tversky_weight
        self.alpha = Tensor(alpha, mindspore.float32)
        self.beta = Tensor(beta, mindspore.float32)
        self.gamma = Tensor(gamma, mindspore.float32)
        self.smooth = Tensor(smooth, mindspore.float32)
        self.one = Tensor(1.0, mindspore.float32)
        self.pow = ops.Pow()

    def construct(self, pred, label, weight_map):
        pred = ops.clip_by_value(pred, self.smooth, self.one - self.smooth)
        bce = -(label * ops.log(pred) + (self.one - label) * ops.log(self.one - pred))
        weighted_bce = ops.sum(weight_map * bce) / (ops.sum(weight_map) + self.smooth)

        tp = ops.sum(pred * label)
        fp = ops.sum(pred * (self.one - label))
        fn = ops.sum((self.one - pred) * label)
        tversky = (tp + self.smooth) / (tp + self.alpha * fp + self.beta * fn + self.smooth)
        focal_tversky = self.pow(self.one - tversky, self.gamma)
        return self.bce_weight * weighted_bce + self.focal_tversky_weight * focal_tversky


loss_fn = PixelWeightMapBCEFocalTverskyLoss(
    bce_weight=WEIGHTED_BCE_WEIGHT,
    focal_tversky_weight=FOCAL_TVERSKY_WEIGHT,
    alpha=TVERSKY_ALPHA,
    beta=TVERSKY_BETA,
    gamma=FOCAL_TVERSKY_GAMMA,
)
print(
    'loss:',
    f'{WEIGHTED_BCE_WEIGHT} * pixel-weight-map BCE + {FOCAL_TVERSKY_WEIGHT} * FocalTversky,',
    f'alpha(FP)={TVERSKY_ALPHA}, beta(FN)={TVERSKY_BETA}, gamma={FOCAL_TVERSKY_GAMMA}',
    f'w0={WEIGHT_MAP_W0}, sigma={WEIGHT_MAP_SIGMA}, pos_base={WEIGHT_MAP_POS_CLASS_WEIGHT}'
)

if USE_PRETRAINED:
    ckpt_candidates = [PRETRAINED_CKPT, FALLBACK_CKPT, WEIGHT_MAP_CKPT]
    ckpt_to_load = next((path for path in ckpt_candidates if path.exists()), None)
    if ckpt_to_load is None:
        raise FileNotFoundError(f'Missing pretrained checkpoint candidates: {ckpt_candidates}')
    load_checkpoint(str(ckpt_to_load), net=net)
    print('loaded pretrained checkpoint:', ckpt_to_load)

train_params = list(net.final.get_parameters()) if TRAIN_FINAL_LAYER_ONLY else list(net.trainable_params())
optimizer = nn.SGD(params=train_params, learning_rate=LR)
print('trainable parameter tensors:', len(train_params))


def train_one_epoch(model, dataset, loss_fn, optimizer):
    def forward_fn(data, label, weight_map):
        pred = model(data)
        loss = loss_fn(pred, label, weight_map)
        return loss, pred

    grad_fn = ops.value_and_grad(forward_fn, None, optimizer.parameters, has_aux=True)

    def train_step(data, label, weight_map):
        (loss, pred), grads = grad_fn(data, label, weight_map)
        loss = ops.depend(loss, optimizer(grads))
        return loss, pred

    model.set_train(True)
    metric = SegmentationMetrics()
    total_loss = 0.0
    size = dataset.get_dataset_size()
    for data, label, weight_map in dataset.create_tuple_iterator():
        loss, pred = train_step(data, label, weight_map)
        total_loss += float(loss.asnumpy())
        metric.update(pred, label)
    result = metric.eval()
    result['loss'] = total_loss / max(size, 1)
    result['composite'] = composite_score(result)
    return result


def validate_fullres(model, data_dir, loss_fn, threshold=0.5):
    model.set_train(False)
    metric = SegmentationMetrics()
    losses = []
    for image_path, mask_path in iter_image_mask_paths(data_dir):
        image, label = load_image_mask(image_path, mask_path)
        prob = predict_overlap_tile_array(model, image, PATCH_SIZE, CENTER_SIZE, TILE_BATCH_SIZE)
        weight_map = make_weight_map(label)
        pred_tensor = Tensor(prob[None, None].astype(np.float32), mindspore.float32)
        label_tensor = Tensor(mask_to_chw(label)[None].astype(np.float32), mindspore.float32)
        weight_tensor = Tensor(weight_to_chw(weight_map)[None].astype(np.float32), mindspore.float32)
        losses.append(float(loss_fn(pred_tensor, label_tensor, weight_tensor).asnumpy()))
        metric.update((prob[None, None] > threshold).astype(np.float32), label_tensor)
    result = metric.eval()
    result['loss'] = float(np.mean(losses)) if losses else 0.0
    result['composite'] = composite_score(result)
    return result


def print_metrics(prefix, metrics):
    print(
        f"{prefix} loss={metrics['loss']:.4f} "
        f"acc={metrics['acc']:.4f} iou={metrics['iou']:.4f} "
        f"dice={metrics['dice']:.4f} sens={metrics['sens']:.4f} "
        f"spec={metrics['spec']:.4f} prec={metrics['precision']:.4f} "
        f"fp={metrics['fp_ratio']:.4f} fn={metrics['fn_ratio']:.4f} "
        f"composite={metrics['composite']:.4f}"
    )


In [ ]:
history = []

baseline_metrics = validate_fullres(net, VAL_DIR, loss_fn, threshold=0.5)
print_metrics('Loaded Best Fullres Val', baseline_metrics)
save_checkpoint(net, str(INITIAL_CKPT))
print(f'Saved initial pretrained checkpoint: {INITIAL_CKPT}')

checkpoint_trackers = {
    'best_iou': {'path': BEST_IOU_CKPT, 'metric': 'iou', 'mode': 'max', 'value': -float('inf'), 'epoch': None},
    'best_dice': {'path': BEST_DICE_CKPT, 'metric': 'dice', 'mode': 'max', 'value': -float('inf'), 'epoch': None},
    'best_precision': {'path': BEST_PRECISION_CKPT, 'metric': 'precision', 'mode': 'max', 'value': -float('inf'), 'epoch': None},
    'best_specificity': {'path': BEST_SPEC_CKPT, 'metric': 'spec', 'mode': 'max', 'value': -float('inf'), 'epoch': None},
    'best_val_loss': {'path': BEST_LOSS_CKPT, 'metric': 'loss', 'mode': 'min', 'value': float('inf'), 'epoch': None},
    'best_composite': {'path': BEST_COMPOSITE_CKPT, 'metric': 'composite', 'mode': 'max', 'value': -float('inf'), 'epoch': None},
}


def is_improved(value, tracker):
    if tracker['mode'] == 'min':
        return value < tracker['value'] - EARLY_STOP_MIN_DELTA
    return value > tracker['value'] + EARLY_STOP_MIN_DELTA


def update_checkpoints(epoch, metrics):
    improved = []
    for name, tracker in checkpoint_trackers.items():
        value = metrics[tracker['metric']]
        if is_improved(value, tracker):
            tracker['value'] = value
            tracker['epoch'] = epoch
            save_checkpoint(net, str(tracker['path']))
            improved.append(f"{name}={value:.4f}")
    if improved:
        print('Saved checkpoints:', ', '.join(improved))
    else:
        print('No tracked metric improved in this epoch.')


if RUN_TRAIN:
    epochs_without_iou_improve = 0
    best_iou_in_run = -float('inf')

    for epoch in range(1, EPOCHS + 1):
        print(f'Epoch [{epoch}/{EPOCHS}]')
        train_metrics = train_one_epoch(net, train_dataset, loss_fn, optimizer)
        val_metrics = validate_fullres(net, VAL_DIR, loss_fn, threshold=0.5)
        print_metrics('Train patch', train_metrics)
        print_metrics('Val fullres', val_metrics)

        row = {'epoch': epoch}
        row.update({f'train_{k}': v for k, v in train_metrics.items()})
        row.update({f'val_{k}': v for k, v in val_metrics.items()})
        history.append(row)

        update_checkpoints(epoch, val_metrics)
        save_checkpoint(net, str(LAST_CKPT))

        if val_metrics['iou'] > best_iou_in_run + EARLY_STOP_MIN_DELTA:
            best_iou_in_run = val_metrics['iou']
            epochs_without_iou_improve = 0
        else:
            epochs_without_iou_improve += 1

        if EARLY_STOP_PATIENCE is not None and epochs_without_iou_improve >= EARLY_STOP_PATIENCE:
            print(f"Early stopping at epoch {epoch}; best_iou_in_run={best_iou_in_run:.4f}")
            break

    if history:
        history_path = OUTPUT_DIR / 'train_history.csv'
        with history_path.open('w', newline='', encoding='utf-8') as f:
            writer = csv.DictWriter(f, fieldnames=list(history[0].keys()))
            writer.writeheader()
            writer.writerows(history)
        print('saved history:', history_path)

        summary_rows = []
        for name, tracker in checkpoint_trackers.items():
            summary_rows.append({
                'checkpoint': name,
                'metric': tracker['metric'],
                'mode': tracker['mode'],
                'best_epoch': tracker['epoch'],
                'best_value': tracker['value'],
                'path': str(tracker['path']),
            })
        summary_path = OUTPUT_DIR / 'checkpoint_summary.csv'
        with summary_path.open('w', newline='', encoding='utf-8') as f:
            writer = csv.DictWriter(f, fieldnames=list(summary_rows[0].keys()))
            writer.writeheader()
            writer.writerows(summary_rows)
        print('saved checkpoint summary:', summary_path)
else:
    print('RUN_TRAIN=False, skip training.')


In [ ]:
if history:
    epochs = [row['epoch'] for row in history]
    plt.figure(figsize=(14, 4))
    plt.subplot(1, 3, 1)
    plt.plot(epochs, [row['train_loss'] for row in history], label='train loss')
    plt.plot(epochs, [row['val_loss'] for row in history], label='val loss')
    plt.xlabel('epoch')
    plt.legend()

    plt.subplot(1, 3, 2)
    plt.plot(epochs, [row['val_iou'] for row in history], label='val IoU')
    plt.plot(epochs, [row['val_dice'] for row in history], label='val Dice')
    plt.plot(epochs, [row['val_composite'] for row in history], label='val composite')
    plt.xlabel('epoch')
    plt.legend()

    plt.subplot(1, 3, 3)
    plt.plot(epochs, [row['val_precision'] for row in history], label='val Precision')
    plt.plot(epochs, [row['val_spec'] for row in history], label='val Specificity')
    plt.plot(epochs, [row['val_sens'] for row in history], label='val Sensitivity')
    plt.xlabel('epoch')
    plt.legend()

    plt.tight_layout()
    curve_path = FIG_DIR / 'training_curves.png'
    plt.savefig(curve_path, dpi=160)
    plt.show()
    print('saved figure:', curve_path)


## 8. 阈值扫描


In [ ]:
THRESHOLD_SCAN_CKPT = BEST_COMPOSITE_CKPT if BEST_COMPOSITE_CKPT.exists() else LAST_CKPT
THRESHOLD_VALUES = np.round(np.arange(0.10, 0.91, 0.02), 2)
THRESHOLD_SCAN_CSV = OUTPUT_DIR / 'threshold_scan.csv'
THRESHOLD_SCAN_FIG = FIG_DIR / 'threshold_scan.png'


def collect_fullres_prediction_cache(model, data_dir):
    model.set_train(False)
    cache = []
    for image_path, mask_path in iter_image_mask_paths(data_dir):
        image, label = load_image_mask(image_path, mask_path)
        prob = predict_overlap_tile_array(model, image, PATCH_SIZE, CENTER_SIZE, TILE_BATCH_SIZE)
        cache.append({'image_path': image_path, 'mask_path': mask_path, 'prob': prob.astype(np.float32), 'label': (label > 127).astype(np.float32)})
    return cache


def evaluate_cache_at_threshold(cache, threshold):
    metric = SegmentationMetrics()
    for item in cache:
        metric.update((item['prob'][None, None] > threshold).astype(np.float32), item['label'][None, None])
    result = metric.eval()
    result['composite'] = composite_score(result)
    return result


if THRESHOLD_SCAN_CKPT.exists():
    load_checkpoint(str(THRESHOLD_SCAN_CKPT), net=net)
    print('loaded checkpoint for threshold scan:', THRESHOLD_SCAN_CKPT)
else:
    print('threshold scan checkpoint not found, use current net:', THRESHOLD_SCAN_CKPT)

val_prediction_cache = collect_fullres_prediction_cache(net, VAL_DIR)
threshold_rows = []
for threshold in THRESHOLD_VALUES:
    metrics = evaluate_cache_at_threshold(val_prediction_cache, float(threshold))
    row = {'threshold': float(threshold)}
    row.update(metrics)
    threshold_rows.append(row)

best_threshold_row = max(threshold_rows, key=lambda row: row['iou'])
best_composite_threshold_row = max(threshold_rows, key=lambda row: row['composite'])
print('best threshold by IoU:', best_threshold_row)
print('best threshold by composite:', best_composite_threshold_row)

with THRESHOLD_SCAN_CSV.open('w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=list(threshold_rows[0].keys()))
    writer.writeheader()
    writer.writerows(threshold_rows)
print('saved threshold scan:', THRESHOLD_SCAN_CSV)

plt.figure(figsize=(9, 5))
plt.plot([r['threshold'] for r in threshold_rows], [r['iou'] for r in threshold_rows], label='IoU')
plt.plot([r['threshold'] for r in threshold_rows], [r['dice'] for r in threshold_rows], label='Dice')
plt.plot([r['threshold'] for r in threshold_rows], [r['precision'] for r in threshold_rows], label='Precision')
plt.plot([r['threshold'] for r in threshold_rows], [r['sens'] for r in threshold_rows], label='Sensitivity')
plt.plot([r['threshold'] for r in threshold_rows], [r['spec'] for r in threshold_rows], label='Specificity')
plt.plot([r['threshold'] for r in threshold_rows], [r['composite'] for r in threshold_rows], label='Composite')
plt.axvline(best_threshold_row['threshold'], color='red', linestyle='--', label=f"best IoU={best_threshold_row['threshold']:.2f}")
plt.axvline(best_composite_threshold_row['threshold'], color='purple', linestyle=':', label=f"best comp={best_composite_threshold_row['threshold']:.2f}")
plt.xlabel('threshold')
plt.ylabel('metric')
plt.legend()
plt.tight_layout()
plt.savefig(THRESHOLD_SCAN_FIG, dpi=160)
plt.show()
print('saved threshold figure:', THRESHOLD_SCAN_FIG)

BEST_THRESHOLD = best_threshold_row['threshold']
BEST_COMPOSITE_THRESHOLD = best_composite_threshold_row['threshold']
print('BEST_THRESHOLD =', BEST_THRESHOLD)
print('BEST_COMPOSITE_THRESHOLD =', BEST_COMPOSITE_THRESHOLD)


## 8. 预测与 mask 可视化

In [ ]:
def preprocess_image(image_path, img_size=None):
    image = np.array(Image.open(image_path).convert('RGB'))
    original_size = (image.shape[1], image.shape[0])
    return image, original_size


def predict_probability_original_size(model, image_path, img_size=None):
    image = np.array(Image.open(image_path).convert('RGB'))
    return predict_overlap_tile_array(model, image, PATCH_SIZE, CENTER_SIZE, TILE_BATCH_SIZE)


def predict_folder(model, image_dir, result_dir, img_size=None, ckpt_path=None, threshold=0.5):
    image_dir = Path(image_dir)
    result_dir = Path(result_dir)
    result_dir.mkdir(parents=True, exist_ok=True)
    prob_dir = result_dir.parent / 'probability'
    prob_dir.mkdir(parents=True, exist_ok=True)
    if ckpt_path and Path(ckpt_path).exists():
        load_checkpoint(str(ckpt_path), net=model)
        print('loaded checkpoint:', ckpt_path)

    model.set_train(False)
    image_paths = sorted(image_dir.glob('*.png'))
    pred_paths = []
    for image_path in image_paths:
        prob = predict_probability_original_size(model, image_path)
        prob_img = Image.fromarray(np.clip(prob * 255.0, 0, 255).astype(np.uint8))
        prob_img.save(prob_dir / image_path.name)

        mask = (prob > threshold).astype(np.uint8) * 255
        mask_img = Image.fromarray(mask)
        out_path = result_dir / image_path.name
        mask_img.save(out_path)
        pred_paths.append(out_path)
    print(f'saved {len(pred_paths)} fullres binary masks to {result_dir} at threshold={threshold}')
    print(f'saved fullres probability maps to {prob_dir}')
    return image_paths, pred_paths


def make_prediction_grid(image_paths, pred_paths, out_file, mask_dir=None, num=6):
    image_paths = list(image_paths)[:num]
    pred_paths = list(pred_paths)[:num]
    rows = 3 if mask_dir else 2
    plt.figure(figsize=(3 * len(image_paths), 3 * rows))
    for i, (image_path, pred_path) in enumerate(zip(image_paths, pred_paths)):
        image = Image.open(image_path).convert('L')
        pred = Image.open(pred_path).convert('L')
        plt.subplot(rows, len(image_paths), i + 1)
        plt.imshow(image, cmap='gray')
        plt.title('image')
        plt.axis('off')
        if mask_dir:
            mask_path = Path(mask_dir) / image_path.name
            plt.subplot(rows, len(image_paths), len(image_paths) + i + 1)
            label_img = Image.open(mask_path).convert('L')
            if INVERT_MASK:
                label_img = Image.fromarray(255 - np.array(label_img))
            plt.imshow(label_img, cmap='gray')
            plt.title('label')
            plt.axis('off')
            pred_row = 2 * len(image_paths) + i + 1
        else:
            pred_row = len(image_paths) + i + 1
        plt.subplot(rows, len(image_paths), pred_row)
        plt.imshow(pred, cmap='gray')
        plt.title('pred')
        plt.axis('off')
    plt.tight_layout()
    plt.savefig(out_file, dpi=160)
    plt.show()
    print('saved grid:', out_file)


In [ ]:
PREDICT_SPLIT = 'val'

if RUN_PREDICT:
    if PREDICT_SPLIT == 'val':
        target_image_dir = VAL_DIR / 'image'
    elif PREDICT_SPLIT == 'test':
        target_image_dir = TEST_DIR / 'image'
    else:
        raise ValueError("PREDICT_SPLIT must be 'val' or 'test'")

    predict_ckpt = HANDOUT_BEST_CKPT if USE_HANDOUT_BEST_FOR_PREDICT and HANDOUT_BEST_CKPT.exists() else BEST_COMPOSITE_CKPT
    if not predict_ckpt.exists():
        predict_ckpt = LAST_CKPT if LAST_CKPT.exists() else INITIAL_CKPT
    print('using prediction checkpoint:', predict_ckpt)
    print('predict split:', PREDICT_SPLIT)
    print('target image dir:', target_image_dir)

    prediction_threshold = BEST_COMPOSITE_THRESHOLD if 'BEST_COMPOSITE_THRESHOLD' in globals() else 0.5
    print('prediction threshold:', prediction_threshold)
    image_paths, pred_paths = predict_folder(net, target_image_dir, PRED_DIR, IMG_SIZE, predict_ckpt, prediction_threshold)
    candidate_mask_dir = target_image_dir.parent / 'mask'
    mask_dir = candidate_mask_dir if candidate_mask_dir.exists() else None
    print('mask dir:', mask_dir if mask_dir else 'None')

    print('image / label / pred mapping:')
    for image_path, pred_path in zip(image_paths, pred_paths):
        label_path = (mask_dir / image_path.name) if mask_dir else None
        label_text = str(label_path) if label_path and label_path.exists() else 'No label'
        print(f'image={image_path} | label={label_text} | pred={pred_path}')

    grid_path = FIG_DIR / f'{PREDICT_SPLIT}_prediction_grid.png'
    make_prediction_grid(image_paths, pred_paths, grid_path, mask_dir=mask_dir, num=6)
else:
    print('RUN_PREDICT=False, skip prediction.')


## 10. Train set 可视化诊断


In [ ]:
def load_boundary_label(mask_path):
    arr = np.asarray(Image.open(mask_path).convert('L'))
    if INVERT_MASK:
        arr = 255 - arr
    return arr > 127


def predict_probability_original_size(model, image_path, img_size=None):
    image = np.array(Image.open(image_path).convert('RGB'))
    return predict_overlap_tile_array(model, image, PATCH_SIZE, CENTER_SIZE, TILE_BATCH_SIZE)


def binary_metrics_from_masks(pred_mask, true_mask, smooth=1e-5):
    pred = pred_mask.reshape(-1).astype(np.float32)
    true = true_mask.reshape(-1).astype(np.float32)
    tp = np.sum(pred * true)
    tn = np.sum((1 - pred) * (1 - true))
    fp = np.sum(pred * (1 - true))
    fn = np.sum((1 - pred) * true)
    union = tp + fp + fn
    return {
        'acc': float((tp + tn) / (tp + tn + fp + fn + smooth)),
        'iou': float(tp / (union + smooth)),
        'dice': float((2 * tp) / (2 * tp + fp + fn + smooth)),
        'sens': float(tp / (tp + fn + smooth)),
        'spec': float(tn / (tn + fp + smooth)),
        'precision': float(tp / (tp + fp + smooth)),
        'fp_ratio': float(fp / (union + smooth)),
        'fn_ratio': float(fn / (union + smooth)),
        'pred_pos_ratio': float(np.mean(pred_mask) / (np.mean(true_mask) + smooth)),
    }


def make_error_map(pred_mask, true_mask):
    error = np.zeros((*true_mask.shape, 3), dtype=np.uint8)
    tp = pred_mask & true_mask
    fp = pred_mask & (~true_mask)
    fn = (~pred_mask) & true_mask
    error[tp] = [0, 220, 0]
    error[fp] = [255, 40, 40]
    error[fn] = [40, 120, 255]
    return error


def save_panel(image_path, label_mask, pred_mask, error_map, out_file):
    image = Image.open(image_path).convert('L')
    plt.figure(figsize=(12, 3))
    items = [
        ('image', image, 'gray'),
        ('label', label_mask, 'gray'),
        ('pred', pred_mask, 'gray'),
        ('error: green TP / red FP / blue FN', error_map, None),
    ]
    for i, (title, arr, cmap) in enumerate(items, start=1):
        plt.subplot(1, 4, i)
        plt.imshow(arr, cmap=cmap)
        plt.title(title)
        plt.axis('off')
    plt.tight_layout()
    plt.savefig(out_file, dpi=160)
    plt.close()


def make_ranked_grid(rows, title, out_file, max_items=5):
    rows = rows[:max_items]
    if not rows:
        return
    plt.figure(figsize=(3 * len(rows), 12))
    for i, row in enumerate(rows):
        image = Image.open(row['image_path']).convert('L')
        label = Image.open(row['label_path']).convert('L')
        pred = Image.open(row['pred_path']).convert('L')
        error = Image.open(row['error_path']).convert('RGB')
        for r, (name, arr, cmap) in enumerate([
            ('image', image, 'gray'),
            ('label', label, 'gray'),
            ('pred', pred, 'gray'),
            ('error', error, None),
        ]):
            plt.subplot(4, len(rows), r * len(rows) + i + 1)
            plt.imshow(arr, cmap=cmap)
            if r == 0:
                plt.title(f"{Path(row['image_path']).name}\nIoU={row['iou']:.3f}")
            else:
                plt.title(name)
            plt.axis('off')
    plt.suptitle(title)
    plt.tight_layout()
    plt.savefig(out_file, dpi=160)
    plt.show()
    print('saved grid:', out_file)


def visualize_train_set(model, ckpt_path, threshold, topk=5):
    image_dir = TRAIN_DIR / 'image'
    mask_dir = TRAIN_DIR / 'mask'
    out_prob_dir = TRAIN_VIZ_DIR / 'probability'
    out_pred_dir = TRAIN_VIZ_DIR / 'pred'
    out_label_dir = TRAIN_VIZ_DIR / 'label'
    out_error_dir = TRAIN_VIZ_DIR / 'error_map'
    out_panel_dir = TRAIN_VIZ_DIR / 'panels'
    for path in [out_prob_dir, out_pred_dir, out_label_dir, out_error_dir, out_panel_dir]:
        path.mkdir(parents=True, exist_ok=True)

    if ckpt_path and Path(ckpt_path).exists():
        load_checkpoint(str(ckpt_path), net=model)
        print('loaded checkpoint for train visualization:', ckpt_path)
    model.set_train(False)

    rows = []
    for image_path in sorted(image_dir.glob('*.png')):
        mask_path = mask_dir / image_path.name
        if not mask_path.exists():
            print('skip image without mask:', image_path)
            continue
        label_mask = load_boundary_label(mask_path)
        prob = predict_probability_original_size(model, image_path, IMG_SIZE)
        pred_mask = prob > threshold
        error_map = make_error_map(pred_mask, label_mask)
        metrics = binary_metrics_from_masks(pred_mask, label_mask)

        prob_path = out_prob_dir / image_path.name
        pred_path = out_pred_dir / image_path.name
        label_path = out_label_dir / image_path.name
        error_path = out_error_dir / image_path.name
        panel_path = out_panel_dir / image_path.name

        Image.fromarray(np.clip(prob * 255.0, 0, 255).astype(np.uint8)).save(prob_path)
        Image.fromarray(pred_mask.astype(np.uint8) * 255).save(pred_path)
        Image.fromarray(label_mask.astype(np.uint8) * 255).save(label_path)
        Image.fromarray(error_map).save(error_path)
        save_panel(image_path, label_mask, pred_mask, error_map, panel_path)

        row = {
            'image': image_path.name,
            'image_path': str(image_path),
            'label_path': str(label_path),
            'pred_path': str(pred_path),
            'prob_path': str(prob_path),
            'error_path': str(error_path),
            'panel_path': str(panel_path),
            'threshold': threshold,
        }
        row.update(metrics)
        rows.append(row)

    if not rows:
        print('No train visualization rows created.')
        return []

    csv_path = TRAIN_VIZ_DIR / 'train_metrics.csv'
    with csv_path.open('w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)
    print('saved train metrics:', csv_path)

    mean_metrics = {k: np.mean([row[k] for row in rows]) for k in ['iou', 'dice', 'sens', 'spec', 'precision', 'fp_ratio', 'fn_ratio', 'pred_pos_ratio']}
    print('train mean metrics:', mean_metrics)

    low_iou = sorted(rows, key=lambda r: r['iou'])
    high_fp = sorted(rows, key=lambda r: r['fp_ratio'], reverse=True)
    high_fn = sorted(rows, key=lambda r: r['fn_ratio'], reverse=True)
    make_ranked_grid(low_iou, 'lowest train IoU samples', TRAIN_VIZ_DIR / 'lowest_iou_grid.png', topk)
    make_ranked_grid(high_fp, 'highest train FP samples', TRAIN_VIZ_DIR / 'highest_fp_grid.png', topk)
    make_ranked_grid(high_fn, 'highest train FN samples', TRAIN_VIZ_DIR / 'highest_fn_grid.png', topk)
    return rows


if RUN_TRAIN_VISUALIZATION:
    train_viz_ckpt = BEST_COMPOSITE_CKPT if BEST_COMPOSITE_CKPT.exists() else LAST_CKPT
    if not train_viz_ckpt.exists():
        train_viz_ckpt = INITIAL_CKPT
    train_viz_threshold = BEST_COMPOSITE_THRESHOLD if 'BEST_COMPOSITE_THRESHOLD' in globals() else 0.5
    print('train visualization checkpoint:', train_viz_ckpt)
    print('train visualization threshold:', train_viz_threshold)
    train_viz_rows = visualize_train_set(net, train_viz_ckpt, train_viz_threshold, TRAIN_VIZ_TOPK)
else:
    print('RUN_TRAIN_VISUALIZATION=False, skip train visualization.')


## 10. 后处理


In [ ]:
POST_THRESHOLD = BEST_THRESHOLD if 'BEST_THRESHOLD' in globals() else 0.68
POST_MIN_AREA = 24
POST_OPEN_ITER = 1
POST_CLOSE_ITER = 0
POST_REMOVE_ROUND_BLOBS = True
POST_ROUND_AREA_MIN = 80
POST_ROUND_CIRCULARITY = 0.35
POST_ROUND_ASPECT_MAX = 2.2
POST_ROUND_EXTENT_MIN = 0.12
POST_DIR = OUTPUT_DIR / 'postprocess'
POST_FIG = FIG_DIR / f'{PREDICT_SPLIT}_postprocess_grid.png'

try:
    from scipy import ndimage
except ImportError:
    ndimage = None


def component_circularity(component):
    area = float(component.sum())
    if area <= 0:
        return 0.0
    eroded = ndimage.binary_erosion(component, structure=np.ones((3, 3), dtype=bool))
    perimeter = float(component.sum() - eroded.sum())
    if perimeter <= 0:
        return 0.0
    return float(4.0 * math.pi * area / (perimeter * perimeter))


def postprocess_mask(mask):
    if ndimage is None:
        raise ImportError('后处理需要 scipy.ndimage，请安装 scipy。')

    mask = mask.astype(bool)
    structure = np.ones((3, 3), dtype=bool)

    if POST_OPEN_ITER > 0:
        mask = ndimage.binary_opening(mask, structure=structure, iterations=POST_OPEN_ITER)
    if POST_CLOSE_ITER > 0:
        mask = ndimage.binary_closing(mask, structure=structure, iterations=POST_CLOSE_ITER)

    labeled, num = ndimage.label(mask, structure=structure)
    cleaned = np.zeros_like(mask, dtype=bool)
    objects = ndimage.find_objects(labeled)

    for idx, slc in enumerate(objects, start=1):
        if slc is None:
            continue
        component = labeled[slc] == idx
        area = int(component.sum())
        if area < POST_MIN_AREA:
            continue

        h = slc[0].stop - slc[0].start
        w = slc[1].stop - slc[1].start
        aspect = max(h / max(w, 1), w / max(h, 1))
        extent = area / max(h * w, 1)
        circularity = component_circularity(component)

        is_round_blob = (
            POST_REMOVE_ROUND_BLOBS
            and area >= POST_ROUND_AREA_MIN
            and circularity >= POST_ROUND_CIRCULARITY
            and aspect <= POST_ROUND_ASPECT_MAX
            and extent >= POST_ROUND_EXTENT_MIN
        )
        if is_round_blob:
            continue

        cleaned[slc] |= component

    return cleaned


def postprocess_prediction_folder(image_dir, pred_dir, out_dir, threshold=0.68):
    image_dir = Path(image_dir)
    pred_dir = Path(pred_dir)
    out_dir = Path(out_dir)
    prob_dir = pred_dir.parent / 'probability'
    out_dir.mkdir(parents=True, exist_ok=True)

    image_paths = sorted(image_dir.glob('*.png'))
    post_paths = []
    for image_path in image_paths:
        prob_path = prob_dir / image_path.name
        pred_path = pred_dir / image_path.name
        if prob_path.exists():
            prob = np.array(Image.open(prob_path).convert('L')).astype(np.float32) / 255.0
            raw_mask = prob > threshold
        elif pred_path.exists():
            raw_mask = np.array(Image.open(pred_path).convert('L')) > 127
        else:
            continue

        post_mask = postprocess_mask(raw_mask)
        out_path = out_dir / image_path.name
        Image.fromarray(post_mask.astype(np.uint8) * 255).save(out_path)
        post_paths.append(out_path)

    print(f'saved {len(post_paths)} postprocessed masks to {out_dir}')
    return image_paths[:len(post_paths)], post_paths


def evaluate_mask_folder(mask_dir, pred_dir):
    metric = SegmentationMetrics()
    mask_dir = Path(mask_dir)
    pred_dir = Path(pred_dir)
    for pred_path in sorted(pred_dir.glob('*.png')):
        label_path = mask_dir / pred_path.name
        if not label_path.exists():
            continue
        pred = np.array(Image.open(pred_path).convert('L')) > 127
        label = np.array(Image.open(label_path).convert('L'))
        if INVERT_MASK:
            label = 255 - label
        label = label > 127
        if pred.shape != label.shape:
            label = np.array(Image.fromarray(label.astype(np.uint8) * 255).resize(pred.shape[::-1], Image.NEAREST)) > 127
        metric.update(pred[None, None, ...].astype(np.float32), label[None, None, ...].astype(np.float32))
    return metric.eval()


def make_postprocess_grid(image_paths, pred_dir, post_paths, out_file, mask_dir=None, num=6):
    image_paths = list(image_paths)[:num]
    post_paths = list(post_paths)[:num]
    rows = 4 if mask_dir else 3
    plt.figure(figsize=(3 * len(image_paths), 3 * rows))
    for i, (image_path, post_path) in enumerate(zip(image_paths, post_paths)):
        pred_path = Path(pred_dir) / image_path.name

        plt.subplot(rows, len(image_paths), i + 1)
        plt.imshow(Image.open(image_path).convert('L'), cmap='gray')
        plt.title('image')
        plt.axis('off')

        row_offset = len(image_paths)
        if mask_dir:
            label_img = Image.open(Path(mask_dir) / image_path.name).convert('L')
            if INVERT_MASK:
                label_img = Image.fromarray(255 - np.array(label_img))
            plt.subplot(rows, len(image_paths), row_offset + i + 1)
            plt.imshow(label_img, cmap='gray')
            plt.title('label')
            plt.axis('off')
            row_offset += len(image_paths)

        plt.subplot(rows, len(image_paths), row_offset + i + 1)
        plt.imshow(Image.open(pred_path).convert('L'), cmap='gray')
        plt.title('pred')
        plt.axis('off')

        plt.subplot(rows, len(image_paths), row_offset + len(image_paths) + i + 1)
        plt.imshow(Image.open(post_path).convert('L'), cmap='gray')
        plt.title('post')
        plt.axis('off')

    plt.tight_layout()
    plt.savefig(out_file, dpi=160)
    plt.show()
    print('saved postprocess grid:', out_file)


if RUN_PREDICT:
    post_image_paths, post_paths = postprocess_prediction_folder(target_image_dir, PRED_DIR, POST_DIR, threshold=POST_THRESHOLD)
    if mask_dir:
        print('before postprocess:', evaluate_mask_folder(mask_dir, PRED_DIR))
        print('after  postprocess:', evaluate_mask_folder(mask_dir, POST_DIR))
    make_postprocess_grid(post_image_paths, PRED_DIR, post_paths, POST_FIG, mask_dir=mask_dir, num=6)
else:
    print('RUN_PREDICT=False, skip postprocess.')


本 notebook 会输出：
- `output_patch_overlap_tile_continue_more/train_history.csv`：继续加训曲线数据。
- `output_patch_overlap_tile_continue_more/checkpoint/*.ckpt`：本轮继续加训独立保存的多种最优 checkpoint。
- `output_patch_overlap_tile_continue_more/checkpoint_summary.csv`：各 checkpoint 对应的指标和 epoch。
- `output_patch_overlap_tile_continue_more/threshold_scan.csv`：基于 fullres overlap-tile 的阈值扫描结果。
- `output_patch_overlap_tile_continue_more/predict/`：fullres overlap-tile 预测 mask。
- `output_patch_overlap_tile_continue_more/train_visualization/`：train set 的 image/label/pred/error map 诊断图。

远程打包命令：
```bash
zip -r unet_patch_overlap_tile_continue_more_results.zip output_patch_overlap_tile_continue_more
```
